# CS156 — Session 21: Assignment 2 Preparation
## Pairwise representation learning for agent-action classification

**PCW source:** submitted *CS156 Session 21* (19 November 2026), Questions 1–4. This notebook accompanies the standalone Python version at `src/session21_assignment2_prep.py`. The **Session 11 housing extension** is in the separate notebook `21_session11_extension_ames.ipynb`.

### Q1 — A model beyond the class: SetFit

SetFit (Sentence Transformer Fine-Tuning) builds on **transfer learning**, **learned embeddings**, and **contrastive learning**. A pretrained sentence transformer is fine-tuned using pairs of labeled text examples; an additional classifier is then fitted on those embeddings. The intended Assignment 2 use case is to represent user objectives, agent tool calls, and permission context, then classify proposed actions as authorized or potentially unsafe.

**Suitable future training data:** labeled agent tool-call events with associated user request and authorization context, with clear provenance and privacy controls. [AgentDojo](https://github.com/ethz-spylab/agentdojo) supplies security scenarios, but is not a drop-in dataset of labeled tool-call actions.

**Evaluation for the future model:** precision, recall, F1, confusion matrix, false-positive rate, and scenario-held-out evaluation. For risk screening, false alarms and missed dangerous actions are both important. **No such generalization test is performed here.**

### Q2 — Two-slide presentation

The in-class presentation introduced (1) the intuition of training on pairs, (2) how a sentence encoder is adapted, (3) a small similarity objective, and (4) why SetFit can work with few labels. The presentation was submitted in the PCW as `Session21_SetFit_Earth_Tones.pdf`. Its full binary is not required to execute the notebooks.

### Q3 — Tools and datasets

- Official [Hugging Face SetFit](https://huggingface.co/docs/setfit/index), [SetFit tutorial](https://huggingface.co/docs/setfit/quickstart), PyTorch, Sentence Transformers, and scikit-learn.
- Public tutorial dataset: [SetFit/sst2](https://huggingface.co/datasets/SetFit/sst2).
- Security scenarios for later: [AgentDojo](https://github.com/ethz-spylab/agentdojo).

### Mini-implementation — not full SetFit

To remain runnable in the PCW's small kernel, we model 16 **synthetic** authorized/unsafe task-action descriptions as normalized bag-of-words vectors `x_i`. We learn a 12-dimensional projection `W` using all 120 unique pairs. Same-label examples have target cosine similarity 1; different-label examples target 0.

$$
\mathcal{L}(W)=\frac{1}{|P|}\sum_{(i,j)\in P}
\left[\frac{(x_iW)^\top(x_jW)}{\|x_iW\|_2\,\|x_jW\|_2}
-\mathbf{1}(y_i=y_j)\right]^2.
$$

**Important:** This is a simplified squared-cosine exercise, **not SetFit's exact default loss**, and it **does not load or fine-tune a transformer, train a prediction head, or assess held-out accuracy**. Good training-pair separation is not evidence of real-world threat detection.

In [1]:
import re
import numpy as np

ALLOWED = [
    "User requests a repo summary. Agent reads README.md.",
    "User requests a repo summary. Agent reads docs/guide.md.",
    "User requests tests. Agent runs pytest.",
    "User requests a code fix. Agent edits src/helper.py.",
    "User requests a report. Agent reads reports/monthly.csv.",
    "User approves staging deployment. Agent deploys to staging.",
    "User requests log review. Agent reads logs/app.log.",
    "User requests a documentation fix. Agent edits docs/setup.md.",
]
UNSAFE = [
    "User requests a repo summary. Agent reads .env secrets.",
    "User requests a repo summary. Agent uploads keys to external host.",
    "User requests tests. Agent deletes the database.",
    "User requests a code fix. Agent edits production permissions.",
    "User requests a report. Agent exports private customer records.",
    "User approves staging deployment. Agent deploys to production.",
    "User requests log review. Agent turns off audit logging.",
    "User requests a documentation fix. Agent modifies admin accounts.",
]

def contrastive_objective(X, W, pairs):
    """Pairwise cosine-MSE teaching loss with analytic gradient."""
    loss = 0.0
    gradient = np.zeros_like(W)
    similarities = {0: [], 1: []}
    for i, j, target in pairs:
        a, b = X[i] @ W, X[j] @ W
        na, nb = np.linalg.norm(a) + 1e-8, np.linalg.norm(b) + 1e-8
        za, zb = a / na, b / nb
        sim = float(za @ zb)
        similarities[int(target)].append(sim)
        error = sim - target
        loss += error ** 2
        grad_a = (zb - sim * za) / na
        grad_b = (za - sim * zb) / nb
        gradient += 2 * error * (
            np.outer(X[i], grad_a) + np.outer(X[j], grad_b)
        )
    n = len(pairs)
    return loss / n, gradient / n, {
        k: float(np.mean(v)) for k, v in similarities.items()
    }

def run_contrastive(steps=300, learning_rate=0.5):
    texts = ALLOWED + UNSAFE
    labels = np.array([0] * len(ALLOWED) + [1] * len(UNSAFE))
    tokens = [set(re.findall(r"[a-z0-9_./-]+", t.lower())) for t in texts]
    vocab = sorted(set().union(*tokens))
    X = np.array([[float(word in row) for word in vocab] for row in tokens])
    X /= np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-8)
    pairs = [(i, j, float(labels[i] == labels[j]))
             for i in range(len(texts)) for j in range(i + 1, len(texts))]
    W = np.random.default_rng(7).normal(0, 0.2, (len(vocab), 12))
    before_loss, _, before = contrastive_objective(X, W, pairs)
    losses = []
    for _ in range(steps):
        loss, gradient, _ = contrastive_objective(X, W, pairs)
        losses.append(loss)
        W -= learning_rate * gradient
    after_loss, _, after = contrastive_objective(X, W, pairs)
    print(f"Synthetic examples: {len(texts)} | unique pairs: {len(pairs)}")
    print(f"Training loss: {before_loss:.3f} -> {after_loss:.3f}")
    print(f"Same-label similarity: {before[1]:.3f} -> {after[1]:.3f}")
    print(f"Different-label similarity: {before[0]:.3f} -> {after[0]:.3f}")
    print("Training pairs only: NOT an unseen security-model evaluation.")
    return {"before": before, "after": after, "losses": losses,
            "initial_loss": before_loss, "final_loss": after_loss}

In [2]:
results = run_contrastive()

Synthetic examples: 16 | unique pairs: 120
Training loss: 0.326 -> 0.000
Same-label similarity: 0.569 -> 0.978
Different-label similarity: 0.616 -> -0.005
Training pairs only: NOT an unseen security-model evaluation.


## Interpretation and next steps

The same-label training-pair cosine similarity increased, and different-label similarity moved toward zero. This verifies that the toy projection can fit its **own 120 training pairs**. Since the examples are extremely small, templated, and reused in every pair, the results say **nothing yet** about generalization.

For Assignment 2, use a pretrained sentence transformer, fine-tune on **training** scenarios only, add a real classifier, and evaluate on distinct held-out agent tasks and attack types. Compare fixed embeddings, conventional TF-IDF classifiers, one-class novelty detection, and fine-tuned embeddings. Keep explicit authorization rules separate from model scores.

**Source:** the student's submitted Session 21 PCW and its locally run NumPy implementation. No claimed production agent-security model exists in this notebook.